# Sync agent skills to the UC volume

Run by the `sync_agent_skills` job (triggered by `run_sync_agent_skills` when a file under `ground-truth-app/agent_skills/` changes).

* `action=mirror`: mirror `source_dir` (git) into `volume_path` (upload new/changed, delete stale). Fails if a skill folder has no `SKILL.md` or its frontmatter `name` differs from the folder name. Sets task value `changed`.
* `action=restart`: stop and start the app (re-runs the active deployment) so AppKit re-reads the volume (it reads skills only at boot). Waits for any in-flight deployment first.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.dropdown("action", "mirror", ["mirror", "restart"])
dbutils.widgets.text("source_dir", "")
dbutils.widgets.text("volume_path", "")
dbutils.widgets.text("app_name", "")

ACTION = dbutils.widgets.get("action")
SOURCE_DIR = dbutils.widgets.get("source_dir").rstrip("/")
VOLUME_PATH = dbutils.widgets.get("volume_path").rstrip("/")
APP_NAME = dbutils.widgets.get("app_name")
print(f"action={ACTION} source_dir={SOURCE_DIR} volume_path={VOLUME_PATH} app_name={APP_NAME}")

In [0]:
import hashlib, io, os, re
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()


def local_files(root: str) -> dict[str, bytes]:
    """Relative path -> bytes for every file under root (skips dotfiles)."""
    out = {}
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        for f in filenames:
            if f.startswith("."):
                continue
            full = os.path.join(dirpath, f)
            with open(full, "rb") as fh:
                out[os.path.relpath(full, root)] = fh.read()
    return out


def volume_tree(root: str) -> tuple[list[str], list[str]]:
    """(relative file paths, relative directory paths) under a volume directory."""
    files, dirs = [], []
    stack = [root]
    while stack:
        cur = stack.pop()
        for e in w.files.list_directory_contents(cur):
            rel = os.path.relpath(e.path, root)
            if e.is_directory:
                dirs.append(rel)
                stack.append(e.path)
            else:
                files.append(rel)
    return files, dirs


def validate(src: dict[str, bytes]) -> None:
    skills = sorted({p.split("/")[0] for p in src if "/" in p})
    loose = [p for p in src if "/" not in p]
    errors = [f"file outside a skill folder: {p}" for p in loose]
    for s in skills:
        body = src.get(f"{s}/SKILL.md")
        if body is None:
            errors.append(f"{s}: missing SKILL.md")
            continue
        m = re.search(r"^name:\s*(\S+)\s*$", body.decode("utf-8"), re.MULTILINE)
        if not m or m.group(1) != s:
            errors.append(f"{s}: frontmatter name {m.group(1) if m else None!r} != folder name")
    if not skills:
        errors.append("no skills found")
    if errors:
        raise ValueError("Invalid agent_skills source: " + "; ".join(errors))
    print(f"validated {len(skills)} skills: {', '.join(skills)}")


if ACTION == "mirror":
    assert SOURCE_DIR and VOLUME_PATH.startswith("/Volumes/"), "source_dir and a /Volumes/ volume_path are required"
    src = local_files(SOURCE_DIR)
    validate(src)

    dst_files, dst_dirs = volume_tree(VOLUME_PATH)
    uploaded, deleted = [], []

    for rel, data in sorted(src.items()):
        target = f"{VOLUME_PATH}/{rel}"
        if rel in dst_files:
            current = w.files.download(target).contents.read()
            if hashlib.sha256(current).digest() == hashlib.sha256(data).digest():
                continue
        w.files.upload(target, io.BytesIO(data), overwrite=True)
        uploaded.append(rel)

    for rel in sorted(set(dst_files) - set(src)):
        w.files.delete(f"{VOLUME_PATH}/{rel}")
        deleted.append(rel)

    src_dirs = {os.path.dirname(p) for p in src} | {p.split("/")[0] for p in src}
    for rel in sorted(set(dst_dirs) - src_dirs, key=lambda p: -p.count("/")):
        w.files.delete_directory(f"{VOLUME_PATH}/{rel}")
        deleted.append(rel + "/")

    # Verify: the volume now matches git exactly.
    after_files, _ = volume_tree(VOLUME_PATH)
    assert sorted(after_files) == sorted(src), f"volume mismatch after sync: {sorted(after_files)} vs {sorted(src)}"

    changed = bool(uploaded or deleted)
    print(f"uploaded={uploaded}\ndeleted={deleted}\nchanged={changed}")
    dbutils.jobs.taskValues.set(key="changed", value="true" if changed else "false")
    dbutils.jobs.taskValues.set(key="uploaded", value=len(uploaded))
    dbutils.jobs.taskValues.set(key="deleted", value=len(deleted))

In [0]:
import time
from databricks.sdk.service.apps import AppDeploymentState


def deployment_in_progress(app_name: str) -> str | None:
    """Id of an in-flight deployment (pending, or newest deployment still IN_PROGRESS)."""
    app = w.apps.get(app_name)
    if app.pending_deployment:
        return app.pending_deployment.deployment_id
    for d in w.apps.list_deployments(app_name):
        if d.status and d.status.state == AppDeploymentState.IN_PROGRESS:
            return d.deployment_id
        break  # newest first; only the latest one matters
    return None


if ACTION == "restart":
    assert APP_NAME, "app_name is required"
    # bundle deploy rolls out the app in parallel with this job; wait until no
    # deployment is in flight so the restart picks up the final deployment.
    deadline = time.time() + 25 * 60
    while (busy := deployment_in_progress(APP_NAME)):
        if time.time() > deadline:
            raise TimeoutError(f"{APP_NAME}: deployment {busy} still in progress after 25 minutes")
        print(f"waiting for in-flight deployment {busy} ...")
        time.sleep(20)

    # Stop + start re-runs the active deployment unchanged (same snapshot and the
    # env_vars bundle deploy attached). Do NOT create a new deployment here: a plain
    # apps deploy has no env_vars, so the build runs with app.yaml NODE_ENV=production
    # and the prebuild client tests fail (React.act missing), leaving the app UNAVAILABLE.
    active = w.apps.get(APP_NAME).active_deployment
    print(f"restarting {APP_NAME} (active deployment {active.deployment_id if active else None})")
    w.apps.stop_and_wait(APP_NAME)
    w.apps.start_and_wait(APP_NAME)  # returns when compute is ACTIVE; the app then rebuilds

    deadline = time.time() + 20 * 60
    while True:
        app = w.apps.get(APP_NAME)
        dep = app.active_deployment or app.pending_deployment
        state = dep.status.state.value if dep and dep.status else None
        if app.app_status.state.value == "RUNNING" and not app.pending_deployment:
            break
        if state == "FAILED":
            raise RuntimeError(f"{APP_NAME}: deployment {dep.deployment_id} failed after restart: {dep.status.message}")
        if time.time() > deadline:
            raise TimeoutError(f"{APP_NAME} not RUNNING 20 minutes after restart (state {app.app_status.state})")
        time.sleep(15)
    print(f"{APP_NAME}: RUNNING; active deployment {app.active_deployment.deployment_id}")

Manual run: `bundle run sync_agent_skills --target <target>` from `ground-truth-app`.